# `08 — Knapsack problem (0/1)`

We study:
- **Problem statement** (Ali Baba 3)
- **DP solution** in **O(NW)**
- **Answer restoration** (argmax / backtracking)
- **Modifications** (as in slides)

Goal:
- Know the DP definition **d[i][w]**
- Know complexity **O(NW)**
- Be able to restore which items were chosen


## `1. Problem statement`

We have:
- Knapsack capacity **W**
- **N** items
- Each item i has:
  - weight **w[i]**
  - cost/value **c[i]**
- Each item is **undividable** and can be taken **at most once** (0/1 knapsack)

Goal:
Choose a subset of items with total weight ≤ W
and **maximize total cost**:

$$
\sum w_{i_k} \le W,\quad \sum c_{i_k} \to \max
$$

In general this is NP-hard, but if weights are integers and W is small,
we can solve in **O(NW)**.

## `2. Why greedy fails`

A natural greedy idea:
- sort by **c / w** (value density)
- take best ratio items first

But for 0/1 knapsack, this can be wrong (slides show a counterexample).
So we need DP.

## `3. DP definition`

Subproblem:
**d[i][w] = maximum total cost using first i items and capacity w**

Meaning:
- items allowed: 0..i-1
- knapsack capacity = w

Base:
- **d[0][w] = 0** (no items → no value)
- **d[i][0] = 0** (capacity 0 → no value)

Transition:
For item (i-1) with weight wi and cost ci:
- If we do NOT take it: d[i-1][w]
- If we take it (only if wi ≤ w): d[i-1][w-wi] + ci

So:

$$
d[i][w] =
\begin{cases}
d[i-1][w], & \text{if } w_i > w \\
\max(d[i-1][w],\ d[i-1][w-w_i]+c_i), & \text{if } w_i \le w
\end{cases}
$$

Answer:
**d[N][W]**

In [1]:
from typing import List


def knapsack_dp_table(
    weights: List[int],
    costs: List[int],
    *,
    W: int,
    verbose: bool = True
) -> List[List[int]]:
    """
    Builds full DP table d of size (N+1) x (W+1).
    d[i][w] = best value using first i items and capacity w.
    """
    n: int = len(weights)
    d: List[List[int]] = [[0] * (W + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        wi: int = weights[i - 1]
        ci: int = costs[i - 1]

        for w in range(0, W + 1):
            d[i][w] = d[i - 1][w]  # not take
            if wi <= w:
                d[i][w] = max(d[i][w], d[i - 1][w - wi] + ci)

        if verbose:
            print("-" * 70)
            print(f"After considering item {i-1} (w={wi}, c={ci})")
            print("w: ", list(range(W + 1)))
            print("d: ", d[i])

    return d


# Example from slides style (weights/costs similar to the table shown)
W = 8
weights = [3, 3, 5, 6]
costs   = [3, 5, 10, 14]

d = knapsack_dp_table(weights, costs, W=W, verbose=True)
print("\nOptimal value d[N][W] =", d[len(weights)][W])

----------------------------------------------------------------------
After considering item 0 (w=3, c=3)
w:  [0, 1, 2, 3, 4, 5, 6, 7, 8]
d:  [0, 0, 0, 3, 3, 3, 3, 3, 3]
----------------------------------------------------------------------
After considering item 1 (w=3, c=5)
w:  [0, 1, 2, 3, 4, 5, 6, 7, 8]
d:  [0, 0, 0, 5, 5, 5, 8, 8, 8]
----------------------------------------------------------------------
After considering item 2 (w=5, c=10)
w:  [0, 1, 2, 3, 4, 5, 6, 7, 8]
d:  [0, 0, 0, 5, 5, 10, 10, 10, 15]
----------------------------------------------------------------------
After considering item 3 (w=6, c=14)
w:  [0, 1, 2, 3, 4, 5, 6, 7, 8]
d:  [0, 0, 0, 5, 5, 10, 14, 14, 15]

Optimal value d[N][W] = 15


## `4. Restoration idea (argmax / backtracking)`

We start from the answer cell **d[N][W]** and go backwards:

For item i-1:
- If **d[i][w] == d[i-1][w]** → item i-1 was NOT taken
- Else → item i-1 WAS taken:
  - record item (i-1)
  - set w = w - weight[i-1]

Continue until i=0.

This restores one optimal subset.

In [2]:
from typing import List


def restore_items(
    d: List[List[int]],
    weights: List[int],
    costs: List[int],
    *,
    W: int,
    verbose: bool = True
) -> List[int]:
    n: int = len(weights)
    w: int = W
    chosen: List[int] = []

    if verbose:
        print("-" * 70)
        print("Restoring chosen items from d[N][W]")
        print(f"Start at i={n}, w={w}, value={d[n][w]}")

    for i in range(n, 0, -1):
        if d[i][w] == d[i - 1][w]:
            if verbose:
                print(f"i={i-1}: NOT taken (value unchanged: {d[i][w]})")
        else:
            chosen.append(i - 1)
            if verbose:
                print(
                    f"i={i-1}: TAKEN  (w={weights[i-1]}, c={costs[i-1]}) "
                    f"move w: {w} -> {w - weights[i-1]}"
                )
            w -= weights[i - 1]

    chosen.reverse()
    return chosen


W = 8
weights = [3, 3, 5, 6]
costs   = [3, 5, 10, 14]

d = knapsack_dp_table(weights, costs, W=W, verbose=False)
chosen = restore_items(d, weights, costs, W=W, verbose=True)

print("\nChosen item indices:", chosen)
print("Chosen weights:", [weights[i] for i in chosen])
print("Chosen costs:  ", [costs[i] for i in chosen])
print("Total weight:", sum(weights[i] for i in chosen))
print("Total cost:  ", sum(costs[i] for i in chosen))

----------------------------------------------------------------------
Restoring chosen items from d[N][W]
Start at i=4, w=8, value=15
i=3: NOT taken (value unchanged: 15)
i=2: TAKEN  (w=5, c=10) move w: 8 -> 3
i=1: TAKEN  (w=3, c=5) move w: 3 -> 0
i=0: NOT taken (value unchanged: 0)

Chosen item indices: [1, 2]
Chosen weights: [3, 5]
Chosen costs:   [5, 10]
Total weight: 8
Total cost:   15


## `5. Complexity`

DP table has (N+1) * (W+1) states.
Each state is computed in O(1).

So:
- **Time:** O(NW)
- **Memory:** O(NW)

This is fast when W is not too large (e.g. up to 10^5 maybe in optimized form).

## `6.1 Unbounded knapsack (infinite copies)`

Modification:
We can use each item (w[i], c[i]) **any number of times**.

DP over capacity only:
- d[w] = best value for capacity w

$$
d[w] = \max_{w_i \le w}(c_i + d[w - w_i])
$$


In [ ]:
from typing import List

def unbounded_knapsack(weights: List[int], costs: List[int], *, W: int) -> int:
    d: List[int] = [0] * (W + 1)
    for w in range(1, W + 1):
        best: int = 0
        for wi, ci in zip(weights, costs):
            if wi <= w:
                best = max(best, ci + d[w - wi])
        d[w] = best
    return d[W]

print(unbounded_knapsack([3, 5], [5, 10], W=10))

20


## `6.2 Subset sum (reachability)`

Question:
Does there exist a subset of weights with sum exactly W?

DP:
d[i][w] = can we make sum w using first i items?

Base:
- d[0][0] = True
- d[0][w>0] = False

Transition:
- ignore item: d[i-1][w]
- take item (if possible): d[i-1][w-wi]

$$
d[i][w] = d[i-1][w] \ or \ d[i-1][w-w_i]
$$


In [4]:
from typing import List

def subset_sum_possible(weights: List[int], *, W: int, verbose: bool = True) -> bool:
    n: int = len(weights)
    d: List[List[bool]] = [[False] * (W + 1) for _ in range(n + 1)]
    d[0][0] = True

    for i in range(1, n + 1):
        wi: int = weights[i - 1]
        for w in range(W + 1):
            d[i][w] = d[i - 1][w]
            if wi <= w:
                d[i][w] = d[i][w] or d[i - 1][w - wi]

        if verbose:
            reachable = [w for w in range(W + 1) if d[i][w]]
            print(f"after i={i} (wi={wi}), reachable sums: {reachable}")

    return d[n][W]

print(subset_sum_possible([3, 3, 5, 6], W=8, verbose=True))

after i=1 (wi=3), reachable sums: [0, 3]
after i=2 (wi=3), reachable sums: [0, 3, 6]
after i=3 (wi=5), reachable sums: [0, 3, 5, 6, 8]
after i=4 (wi=6), reachable sums: [0, 3, 5, 6, 8]
True


## `6.3 Closest sum ≤ W`

Find W* ≤ W such that:
- W* is reachable by subset sum
- W - W* is minimal

Approach:
- compute subset sum table
- scan w from W down to 0 and find the first reachable w

In [ ]:
from typing import List


def closest_subset_sum(weights: List[int], *, W: int) -> int:
    n: int = len(weights)
    d: List[List[bool]] = [[False] * (W + 1) for _ in range(n + 1)]
    d[0][0] = True

    for i in range(1, n + 1):
        wi: int = weights[i - 1]
        for w in range(W + 1):
            d[i][w] = d[i - 1][w] or (wi <= w and d[i - 1][w - wi])

    for w in range(W, -1, -1):
        if d[n][w]:
            return w
    return 0

print("Closest reachable to 8:", closest_subset_sum([3, 3, 5, 6], W=8))


Closest reachable to 8: 8


## **`Knapsack — Complexity summary (classic oral exam)`**

| Variant | Decision | DP state | Time | Memory | Notes |
|---|---|---|---:|---:|---|
| **0/1 Knapsack** | take item 0 or 1 time | d[i][w] | **O(NW)** | O(NW) | can restore items |
| **0/1 Knapsack (1D memory)** | take item 0 or 1 time | d[w] | **O(NW)** | **O(W)** | loop w descending |
| **Unbounded Knapsack** | take item many times | d[w] | **O(NW)** | O(W) | loop w ascending |
| **Subset Sum** | reachable? True/False | d[i][w] | **O(NW)** | O(NW) | boolean DP |
| **Closest ≤ W** | best reachable weight | d[i][w] | **O(NW)** | O(NW) | scan last row |


## 7. **Real-world mental model**

### **`0/1 Knapsack (most common)`**
You have a limited resource (**capacity W**) and each option has:
- **cost / value** (benefit)
- **weight / size** (resource usage)
Each option can be chosen **at most once**.

Examples:
- **Laptop backpack packing**: weight limit + value of items.
- **Budgeted project selection**: limited budget W, each project has cost and expected return.
- **Cargo loading**: limited volume/weight, each item has value.

### **`Subset Sum`**
Examples:
- Can I pay exactly **W** using a subset of available banknotes?
- Can I partition tasks into exactly **W** hours?

### **`Unbounded Knapsack`**
Examples:
- Buy unlimited units of products (each with cost/benefit) under a budget.
- Manufacturing where you can produce unlimited quantity of each item type.